# Experiment 4: Handling Imbalanced Data

Reconstructed from screenshots. Trains a Random Forest with TF-IDF (trigrams) across several class-imbalance handling strategies, logging each run to MLflow.

**Note:** the confusion matrix save/log lines at the end of Step 6 were cut off in the original screenshot — I've filled those in with the standard MLflow pattern (save figure -> log as artifact). Double check against the source if you get access to it later.

In [15]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

In [16]:
import os
print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir('.'))

Current working directory: g:\Telegram Desktop\YouTube Comment Analysis 💜\yt_comment_analysis\notebooks
Files in current directory: ['.gitkeep', 'baseline_analysis.md', 'confusion_matrix.png', 'confusion_matrix_adasyn.png', 'confusion_matrix_class_weights.png', 'confusion_matrix_oversampling.png', 'confusion_matrix_smote_enn.png', 'confusion_matrix_undersampling.png', 'dataset.csv', 'eda.ipynb', 'experiment-4-handling-imbalanced-data.ipynb', 'experiment_1_baseline_model.ipynb', 'experiment_2_bow_tfidf.ipynb', 'experiment_3_tfidf_(1,3)_max_features.ipynb']


In [17]:
# MLflow / DagsHub tracking setup
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"  # replace before running

mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("imbalance_handling_bigram_issue_updated")

2026/07/20 23:37:51 INFO mlflow.tracking.fluent: Experiment with name 'imbalance_handling_bigram_issue_updated' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/d43f939b6f824052ba10212be9325a3b', creation_time=1784569072267, effective_trace_archival_retention=None, experiment_id='8', last_update_time=1784569072267, lifecycle_stage='active', name='imbalance_handling_bigram_issue_updated', tags={}, trace_location=None, workspace='default'>

In [18]:
# Load your cleaned dataset here
df = pd.read_csv("dataset.csv").dropna(subset=['clean_comment'])
# df should have columns: 'clean_comment' and 'category' (-1, 0, 1)

In [19]:
print(df['clean_comment'].isna().sum())

0


In [ ]:
# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 2)  # Bigram setting
    max_features = 10000  # Set max_features to 1000 for TF-IDF

    # Step 2: Vectorization using TF-IDF
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X = vectorizer.fit_transform(df['clean_comment'])
    y = df['category']

    # Step 3: Handle class imbalance based on the selected method
    if imbalance_method == 'class_weights':
        # Use class_weight in Random Forest
        class_weight = 'balanced'
    else:
        class_weight = None  # Do not apply class_weight if using resampling

    # Resampling Techniques
    if imbalance_method == 'oversampling':
        smote = SMOTE(random_state=42)
        X, y = smote.fit_resample(X, y)
    elif imbalance_method == 'adasyn':
        adasyn = ADASYN(random_state=42)
        X, y = adasyn.fit_resample(X, y)
    elif imbalance_method == 'undersampling':
        rus = RandomUnderSampler(random_state=42)
        X, y = rus.fit_resample(X, y)
    elif imbalance_method == 'smote_enn':
        smote_enn = SMOTEENN(random_state=42)
        X, y = smote_enn.fit_resample(X, y)

    # Step 4: Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Step 5: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag(
            "description",
            f"RandomForest with TF-IDF Bigrams, imbalance handling method={imbalance_method}"
        )

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        # Initialize and train the model
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            class_weight=class_weight
        )
        model.fit(X_train, y_train)

        # Step 6: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {imbalance_method}")

        # --- reconstructed tail (cut off in screenshot) ---
        conf_matrix_path = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig(conf_matrix_path)
        mlflow.log_artifact(conf_matrix_path)
        plt.close()
        # ----------------------------------------------------

        print(f"Completed run for imbalance method: {imbalance_method}, Accuracy: {accuracy:.4f}")

In [21]:
print(df['clean_comment'].isna().sum())

0


In [22]:
# Step 7: Run experiments for different imbalance methods
imbalance_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']

for method in imbalance_methods:
    run_imbalanced_experiment(method)

Completed run for imbalance method: class_weights, Accuracy: 0.6761
🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/8/runs/e719d0a8ab454f74ab2ca031b9a4f9cb
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/8
Completed run for imbalance method: oversampling, Accuracy: 0.7008
🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/8/runs/728349c134d4422b9a6030865ada8013
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/8
Completed run for imbalance method: adasyn, Accuracy: 0.7383
🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/8/runs/56cdeb07804a498797c9ca5d24b6ba3d
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/8
Compl